In [ ]:
import re
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

In [ ]:
data = [
    ("How can I reset my password?",
     "You can reset your password through account settings."),

    ("Where is my order?",
     "You can track your order using the tracking number."),

    ("My order is delayed",
     "We apologize for the delay. Please check your order status."),

    ("How can I contact support?",
     "You can contact our support team through live chat."),

    ("Can I cancel my order?",
     "You can cancel your order before it is shipped."),

    ("How do I change my email?",
     "You can change your email address in account settings."),

    ("I cannot log into my account",
     "Please check your password and try again."),

    ("How can I get a refund?",
     "You can request a refund through your order page."),

    ("Where can I find my invoice?",
     "Your invoice is available in the order history."),

    ("How long does delivery take?",
     "Delivery usually takes three to five business days."),

    ("Can I change my delivery address?",
     "You can update your delivery address before shipment."),

    ("My payment was declined",
     "Please check your payment details or try another card."),

    ("How do I delete my account?",
     "You can delete your account in privacy settings."),

    ("Is customer support available?",
     "Our support team is available every day."),

    ("How can I return a product?",
     "You can start a return through your order page.")
]

df = pd.DataFrame(data, columns=["question", "answer"])

df

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["question_clean"] = df["question"].apply(preprocess_text)
df["answer_clean"] = df["answer"].apply(preprocess_text)

df[["question_clean", "answer_clean"]]

In [ ]:
special_tokens = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]

all_texts = (
    df["question_clean"].tolist() +
    df["answer_clean"].tolist()
)

vocab = set()

for text in all_texts:
    vocab.update(text.split())

word_to_idx = {
    token: idx for idx, token in enumerate(special_tokens)
}

for word in sorted(vocab):
    if word not in word_to_idx:
        word_to_idx[word] = len(word_to_idx)

idx_to_word = {
    idx: word for word, idx in word_to_idx.items()
}

vocab_size = len(word_to_idx)

print("Vocabulary size:", vocab_size)
print(word_to_idx)

In [ ]:
question_tokens = [
    text.split() for text in df["question_clean"]
]

answer_tokens = [
    ["<SOS>"] + text.split() + ["<EOS>"]
    for text in df["answer_clean"]
]

max_input_len = max(len(tokens) for tokens in question_tokens)
max_output_len = max(len(tokens) for tokens in answer_tokens)

print("Max input length:", max_input_len)
print("Max output length:", max_output_len)

In [ ]:
PAD_IDX = word_to_idx["<PAD>"]
UNK_IDX = word_to_idx["<UNK>"]
SOS_IDX = word_to_idx["<SOS>"]
EOS_IDX = word_to_idx["<EOS>"]


def encode_text(tokens, max_len):
    indexes = [
        word_to_idx.get(token, UNK_IDX)
        for token in tokens
    ]

    indexes = indexes[:max_len]

    while len(indexes) < max_len:
        indexes.append(PAD_IDX)

    return indexes


X = [
    encode_text(tokens, max_input_len)
    for tokens in question_tokens
]

Y = [
    encode_text(tokens, max_output_len)
    for tokens in answer_tokens
]

X = torch.tensor(X, dtype=torch.long)
Y = torch.tensor(Y, dtype=torch.long)

print("Questions shape:", X.shape)
print("Answers shape:", Y.shape)

In [ ]:
class SupportDataset(Dataset):
    def __init__(self, questions, answers):
        self.questions = questions
        self.answers = answers

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        return self.questions[idx], self.answers[idx]


dataset = SupportDataset(X, Y)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

In [ ]:
class Seq2SeqRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size=128):
        super().__init__()

        self.encoder = nn.Embedding(
            vocab_size,
            hidden_size,
            padding_idx=PAD_IDX
        )

        self.decoder = nn.Embedding(
            vocab_size,
            hidden_size,
            padding_idx=PAD_IDX
        )

        self.rnn_encoder = nn.RNN(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.rnn_decoder = nn.RNN(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, questions, answers):
        encoder_emb = self.encoder(questions)

        _, hidden = self.rnn_encoder(encoder_emb)

        decoder_input = answers[:, :-1]
        decoder_emb = self.decoder(decoder_input)

        decoder_output, _ = self.rnn_decoder(
            decoder_emb,
            hidden
        )

        return self.fc(decoder_output)


rnn_model = Seq2SeqRNN(vocab_size).to(device)

print(rnn_model)

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

optimizer = optim.Adam(
    rnn_model.parameters(),
    lr=0.001
)

rnn_losses = []

for epoch in range(10):
    rnn_model.train()
    total_loss = 0

    for questions, answers in loader:
        questions = questions.to(device)
        answers = answers.to(device)

        optimizer.zero_grad()

        outputs = rnn_model(questions, answers)

        targets = answers[:, 1:]

        loss = criterion(
            outputs.reshape(-1, vocab_size),
            targets.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    rnn_losses.append(avg_loss)

    print(f"Epoch {epoch + 1}/10 | Loss: {avg_loss:.4f}")

In [ ]:
class Seq2SeqLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size=128):
        super().__init__()

        self.encoder = nn.Embedding(
            vocab_size,
            hidden_size,
            padding_idx=PAD_IDX
        )

        self.decoder = nn.Embedding(
            vocab_size,
            hidden_size,
            padding_idx=PAD_IDX
        )

        self.lstm_encoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.lstm_decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, questions, answers):
        encoder_emb = self.encoder(questions)

        _, (hidden, cell) = self.lstm_encoder(
            encoder_emb
        )

        decoder_input = answers[:, :-1]
        decoder_emb = self.decoder(decoder_input)

        decoder_output, _ = self.lstm_decoder(
            decoder_emb,
            (hidden, cell)
        )

        return self.fc(decoder_output)


lstm_model = Seq2SeqLSTM(vocab_size).to(device)

print(lstm_model)

In [ ]:
lstm_optimizer = optim.Adam(
    lstm_model.parameters(),
    lr=0.001
)

lstm_losses = []

for epoch in range(10):
    lstm_model.train()
    total_loss = 0

    for questions, answers in loader:
        questions = questions.to(device)
        answers = answers.to(device)

        lstm_optimizer.zero_grad()

        outputs = lstm_model(questions, answers)

        targets = answers[:, 1:]

        loss = criterion(
            outputs.reshape(-1, vocab_size),
            targets.reshape(-1)
        )

        loss.backward()
        lstm_optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    lstm_losses.append(avg_loss)

    print(f"Epoch {epoch + 1}/10 | Loss: {avg_loss:.4f}")

In [ ]:
def generate_answer(model, question, model_type="rnn"):
    model.eval()

    question = preprocess_text(question)
    tokens = question.split()

    encoded = encode_text(tokens, max_input_len)

    question_tensor = torch.tensor(
        [encoded],
        dtype=torch.long
    ).to(device)

    generated_words = []

    with torch.no_grad():
        encoder_emb = model.encoder(question_tensor)

        if model_type == "rnn":
            _, hidden = model.rnn_encoder(encoder_emb)
        else:
            _, (hidden, cell) = model.lstm_encoder(encoder_emb)

        current_token = torch.tensor(
            [[SOS_IDX]],
            dtype=torch.long
        ).to(device)

        for _ in range(max_output_len):
            decoder_emb = model.decoder(current_token)

            if model_type == "rnn":
                output, hidden = model.rnn_decoder(
                    decoder_emb,
                    hidden
                )
            else:
                output, (hidden, cell) = model.lstm_decoder(
                    decoder_emb,
                    (hidden, cell)
                )

            prediction = model.fc(output[:, -1, :])
            predicted_idx = prediction.argmax(dim=1).item()

            if predicted_idx in (EOS_IDX, PAD_IDX):
                break

            if predicted_idx in (SOS_IDX, UNK_IDX):
                break

            generated_words.append(
                idx_to_word[predicted_idx]
            )

            current_token = torch.tensor(
                [[predicted_idx]],
                dtype=torch.long
            ).to(device)

    return " ".join(generated_words)

In [ ]:
test_questions = [
    "How can I reset my password?",
    "My order is delayed",
    "How can I get a refund?",
    "I cannot log into my account",
    "How can I contact support?"
]

results = []

for question in test_questions:
    rnn_answer = generate_answer(
        rnn_model, question, "rnn"
    )

    lstm_answer = generate_answer(
        lstm_model, question, "lstm"
    )

    results.append({
        "Question": question,
        "RNN Answer": rnn_answer,
        "LSTM Answer": lstm_answer
    })

results_df = pd.DataFrame(results)

results_df

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    range(1, 11),
    rnn_losses,
    marker="o",
    label="RNN"
)

plt.plot(
    range(1, 11),
    lstm_losses,
    marker="s",
    label="LSTM"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("RNN vs LSTM Training Loss")

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def token_accuracy(prediction, reference):
    pred_tokens = prediction.split()
    ref_tokens = preprocess_text(reference).split()

    if len(ref_tokens) == 0:
        return 0.0

    matches = sum(
        1 for word in pred_tokens
        if word in ref_tokens
    )

    return matches / len(ref_tokens)


reference_answers = [
    data[0][1],
    data[2][1],
    data[7][1],
    data[6][1],
    data[3][1]
]

rnn_scores = []
lstm_scores = []

for question, reference in zip(
    test_questions,
    reference_answers
):
    rnn_prediction = generate_answer(
        rnn_model, question, "rnn"
    )

    lstm_prediction = generate_answer(
        lstm_model, question, "lstm"
    )

    rnn_scores.append(
        token_accuracy(rnn_prediction, reference)
    )

    lstm_scores.append(
        token_accuracy(lstm_prediction, reference)
    )


print("RNN average token accuracy:",
      round(sum(rnn_scores) / len(rnn_scores), 4))

print("LSTM average token accuracy:",
      round(sum(lstm_scores) / len(lstm_scores), 4))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

transformer_tokenizer = AutoTokenizer.from_pretrained(model_name)

transformer_model = AutoModelForCausalLM.from_pretrained(
    model_name
).to(device)

transformer_tokenizer.pad_token = transformer_tokenizer.eos_token

print("DistilGPT2 loaded successfully")

In [ ]:
def generate_transformer_answer(question):
    prompt = (
        "Customer support conversation.\n"
        f"Customer: {question}\n"
        "Support agent:"
    )

    inputs = transformer_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = transformer_model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=transformer_tokenizer.eos_token_id
        )

    generated_text = transformer_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    answer = generated_text[len(prompt):].strip()

    return answer

In [ ]:
transformer_questions = [
    "I forgot my password",
    "My order is delayed",
    "I want to get a refund"
]

for question in transformer_questions:
    answer = generate_transformer_answer(question)

    print("Customer:", question)
    print("SupportFlow AI:", answer)
    print("-" * 50)